<a href="https://colab.research.google.com/github/Himanshu530-collab/BlogSphere/blob/main/model_working_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()


TypeError: 'NoneType' object is not subscriptable

In [ ]:
import zipfile
import os

zip_path = "DentalProblemDetection.zip"  # Make sure this matches exactly
extract_dir = "dental_data"  # Folder to extract into

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

root_dir = "dental_data"
print("Classes:", os.listdir(root_dir))  # should list folders like 'cavity', 'healthy'


In [ ]:
import shutil
import os

inner_dir = "dental_data/DentalProblemDetection"
target_dir = "dental_data"

# Move all subfolders from inner_dir to target_dir
for folder_name in os.listdir(inner_dir):
    full_src_path = os.path.join(inner_dir, folder_name)
    full_dst_path = os.path.join(target_dir, folder_name)
    shutil.move(full_src_path, full_dst_path)

# Remove the now-empty 'DentalProblemDetection' folder
os.rmdir(inner_dir)

# Recheck class folders
print("✅ Fixed Classes:", os.listdir(target_dir))


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import torchvision


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomAffine(0, translate=(0.2, 0.2)),  # ±20% width/height shift
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),  # Converts [0–255] to [0–1]
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Now in [-1, 1]
])


In [ ]:
print("Classes in rgb_images:", os.listdir('dental_data/data/rgb_images'))


In [ ]:
'''import os

healthy_dir = 'dental_data/data/rgb_images/healthy'
os.makedirs(healthy_dir, exist_ok=True)

for filename in uploaded.keys():
    with open(os.path.join(healthy_dir, filename), 'wb') as f:
        f.write(uploaded[filename])'''


In [ ]:
import os
import shutil

# Path to your rgb_images folder
rgb_images_dir = 'dental_data/data/rgb_images'

# List of empty folders to remove
folders_to_delete = ['gum_disease', 'misalignment','healthy']

for folder in folders_to_delete:
    folder_path = os.path.join(rgb_images_dir, folder)
    if os.path.exists(folder_path):
        # Remove the folder and all its contents (if any)
        shutil.rmtree(folder_path)
        print(f"Deleted folder: {folder_path}")
    else:
        print(f"Folder not found (already deleted?): {folder_path}")


In [ ]:
!pip install ultralytics


In [ ]:
'''from ultralytics import YOLO
import cv2
import os

# Load a pretrained YOLOv8 model (you can train your own if you have a dataset)
# Here, we'll try 'yolov8n.pt' (nano model) for demo - replace with your trained weights if any
model = YOLO('yolov8n.pt')

input_dir = 'dental_data/data/rgb_images/healthy'
cropped_dir = 'dental_data/data/rgb_images/healthy_cropped'
os.makedirs(cropped_dir, exist_ok=True)

for img_name in os.listdir(input_dir):
    img_path = os.path.join(input_dir, img_name)
    img = cv2.imread(img_path)

    # Run detection
    results = model(img)

    # results[0].boxes.xyxy gives bounding boxes
    boxes = results[0].boxes.xyxy.cpu().numpy()  # shape: (num_boxes, 4)

    if len(boxes) > 0:
        # Assuming the first detected box is the tooth region
        x1, y1, x2, y2 = boxes[0].astype(int)

        # Crop the image
        cropped_img = img[y1:y2, x1:x2]

        # Save cropped image
        cv2.imwrite(os.path.join(cropped_dir, img_name))
    else:
        # No detection, save original or skip
        print(f"No tooth detected in {img_name}, saving original.")
        cv2.imwrite(os.path.join(cropped_dir, img_name), img)

print("Cropping done!")'''


In [ ]:
!pip install --upgrade ultralytics


In [ ]:
from ultralytics import YOLO
import cv2
import os

model = YOLO('yolov8n.pt')  # small pretrained model

img_path = 'dental_data/data/rgb_images/cavity/0.jpg'  # example image path
img = cv2.imread(img_path)

results = model(img)

print(results)  # check detection output


In [ ]:
import os
from ultralytics import YOLO
from PIL import Image

# Load your YOLO model
model = YOLO('yolov8n.pt')  # change if you use a custom model

root_dir = 'dental_data/data/rgb_images/'
valid_exts = ('.jpg', '.jpeg', '.png')

# Base output directory for annotated images
output_base_dir = 'runs/detect/custom_outputs'

# Make sure base output directory exists
os.makedirs(output_base_dir, exist_ok=True)

for cls_name in os.listdir(root_dir):
    class_input_dir = os.path.join(root_dir, cls_name)
    if not os.path.isdir(class_input_dir):
        print(f"⚠️ Skipping non-directory: {class_input_dir}")
        continue

    # Create corresponding output folder for the class
    class_output_dir = os.path.join(output_base_dir, cls_name)
    os.makedirs(class_output_dir, exist_ok=True)

    print(f"\n📁 Processing class '{cls_name}', saving to '{class_output_dir}'")

    for img_file in os.listdir(class_input_dir):
        if img_file.lower().endswith(valid_exts):
            img_path = os.path.join(class_input_dir, img_file)

            # Run YOLO inference
            results = model(img_path)

            # Get annotated image (numpy array) from results
            annotated_img = results[0].plot()

            # Save annotated image to the class output folder
            save_path = os.path.join(class_output_dir, img_file)
            Image.fromarray(annotated_img).save(save_path)

            print(f"✅ Saved: {save_path}")


In [ ]:
import os

output_base_dir = 'runs/detect/custom_outputs'

for cls_name in os.listdir(output_base_dir):
    class_output_dir = os.path.join(output_base_dir, cls_name)
    if os.path.isdir(class_output_dir):
        print(f"Contents of '{class_output_dir}':")
        print(os.listdir(class_output_dir))
        print()  # blank line for readability


In [ ]:
import os

# For mouth_ulcer folder
mouth_ulcer_files = os.listdir('runs/detect/custom_outputs/mouth_ulcer')
print("Contents of 'runs/detect/custom_outputs/mouth_ulcer':")
print(mouth_ulcer_files)

# For hypodontia folder
hypodontia_files = os.listdir('runs/detect/custom_outputs/hypodontia')
print("\nContents of 'runs/detect/custom_outputs/hypodontia':")
print(hypodontia_files)


In [ ]:
# import os
# from IPython.display import Image, display

# output_dir = 'runs/detect/custom_outputs'
# files = [f for f in os.listdir(output_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# print(f"Found {len(files)} annotated images.")

# # Display first 5 annotated images
# for f in files[:5]:
#     display(Image(filename=os.path.join(output_dir, f)))


In [ ]:
# import os

# print("All folders under 'runs/detect/':")
# print(os.listdir('runs/detect'))


In [ ]:
import glob
from IPython.display import Image, display

for img_path in glob.glob('runs/detect/custom_outputs/**/*.jpg', recursive=True)[:5]:
    display(Image(filename=img_path))



In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

# Change this to your actual results folder
result_dir = 'runs/detect/custom_outputs/tartar'

# List all jpg and png images in the directory
image_files = [f for f in os.listdir(result_dir) if f.endswith(('.jpg', '.png'))]

# Show first 5 images
for image_name in image_files[:5]:
    img_path = os.path.join(result_dir, image_name)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Failed to load {img_path}")
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    plt.title(image_name)
    plt.axis('off')
    plt.show()


In [ ]:
import os
for cls in os.listdir('runs/detect/custom_outputs'):
    count = len(os.listdir(f'runs/detect/custom_outputs/{cls}'))
    print(f"Class {cls}: {count} images processed")


In [ ]:
import os, csv
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
root = 'dental_data/data/rgb_images'
valid_exts = ('.jpg','.jpeg','.png')

with open('results_summary.csv','w',newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['image','class','confidence','xmin','ymin','xmax','ymax'])

    for cls in os.listdir(root):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir): continue
        for img in os.listdir(cls_dir):
            if not img.lower().endswith(valid_exts): continue
            path = os.path.join(cls_dir, img)
            res = model(path)[0]
            for box in res.boxes:
                c = model.names[int(box.cls[0])]
                conf = float(box.conf[0])
                xmin,ymin,xmax,ymax = box.xyxy[0].cpu().numpy()
                writer.writerow([img, c, conf, xmin,ymin,xmax,ymax])
print("✅ CSV summary created.")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('results_summary.csv')

plt.figure(figsize=(10,4))
sns.countplot(data=df, x='class')
plt.title('Detected Objects per Class')
plt.show()

plt.figure(figsize=(10,4))
sns.histplot(df['confidence'], bins=20, kde=True)
plt.title('Confidence Score Distribution')
plt.show()


In [ ]:
df['width']  = df['xmax'] - df['xmin']
df['height'] = df['ymax'] - df['ymin']

plt.figure(figsize=(8,4))
sns.histplot(df['width'], bins=30)
plt.title('Bounding Box Width Distribution')
plt.show()

plt.figure(figsize=(8,4))
sns.histplot(df['height'], bins=30)
plt.title('Bounding Box Height Distribution')
plt.show()


In [ ]:
import glob
from IPython.display import Image, display

for path in glob.glob('runs/detect/custom_outputs/**/*.jpg', recursive=True)[:5]:
    display(Image(filename=path))


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

source_dir = '/content/dental_data/data/rgb_images'
output_base = '/content/split_data'

# Train/val/test folders
train_dir = os.path.join(output_base, 'train')
val_dir = os.path.join(output_base, 'val')
test_dir = os.path.join(output_base, 'test')

# Create directory structure
for dir_path in [train_dir, val_dir, test_dir]:
    for class_name in os.listdir(source_dir):
        os.makedirs(os.path.join(dir_path, class_name), exist_ok=True)

# Split each class
for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)
    images = os.listdir(class_path)

    train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

    for img in train_imgs:
        shutil.copy(os.path.join(class_path, img), os.path.join(train_dir, class_name, img))
    for img in val_imgs:
        shutil.copy(os.path.join(class_path, img), os.path.join(val_dir, class_name, img))
    for img in test_imgs:
        shutil.copy(os.path.join(class_path, img), os.path.join(test_dir, class_name, img))

print("✅ Dataset split into train/val/test.")


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch import nn, optim

# Transforms for training and validation
transform = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
}

# Load datasets
data_dir = output_base
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), transform[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=32, shuffle=True)
               for x in ['train', 'val']}
class_names = image_datasets['train'].classes

# Load pretrained ResNet
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))  # Custom classifier

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

# Define criterion, optimizer, and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)  # LR decay every 3 epochs

num_epochs = 10
patience = 3
best_val_loss = float('inf')
patience_counter = 0

best_model_wts = copy.deepcopy(model.state_dict())

# Initialize lists for tracking
train_loss_history = []
val_loss_history = []
train_acc_history = []
val_acc_history = []

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])
        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        # Track loss and accuracy
        if phase == 'train':
            train_loss_history.append(epoch_loss)
            train_acc_history.append(epoch_acc.item())
        else:
            val_loss_history.append(epoch_loss)
            val_acc_history.append(epoch_acc.item())

            # Save best model
            if epoch_loss < best_val_loss:
                best_val_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                patience_counter = 0
                print("✅ Best model updated.")
            else:
                patience_counter += 1
                print(f"⏳ Early stopping patience: {patience_counter}/{patience}")
                if patience_counter >= patience:
                    print("⛔ Early stopping triggered.")
                    model.load_state_dict(best_model_wts)
                    break

    if patience_counter >= patience:
        break

    scheduler.step()  # Adjust LR after each epoch

# Load best model weights
model.load_state_dict(best_model_wts)
print("🎯 Loaded best model weights.")


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_loss_history) + 1)

plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, train_acc_history, label='Train Accuracy')
plt.plot(epochs, val_acc_history, label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, train_loss_history, label='Train Loss')
plt.plot(epochs, val_loss_history, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform['val'])
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"✅ Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Get all predictions and labels
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Class names (in order of folders in your test directory)
class_names = test_dataset.classes

# Classification Report
print("📊 Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import torch

model.eval()  # set to eval mode
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:  # your test_loader DataLoader
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Overall accuracy
accuracy = (np.array(all_preds) == np.array(all_labels)).mean()
print(f"Test Accuracy: {accuracy*100:.2f}%")

# Detailed report
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)


In [ ]:
torch.save(model.state_dict(), 'best_model.pth')


In [ ]:
from google.colab import files
files.download('best_model.pth')


In [ ]:
# from ultralytics import YOLO

# model = YOLO('yolov8n.pt')

# # Example image from a non-'healthy' folder
# img_path = 'dental_data/data/rgb_images/cavity/1.jpg'  # update with an actual image path

# results = model(img_path)

# # Print what was detected
# results[0].show()
# print(results[0].boxes.cls)  # should show class indices


In [ ]:
# import os
# print(os.getcwd())
# # print(os.listdir('.'))


In [ ]:
# from ultralytics import YOLO
# import os

# model = YOLO('yolov8n.pt')
# folder = 'dental_data/data/rgb_images/cavity'

# for img_name in os.listdir(folder):
#     if img_name.endswith(('.jpg', '.jpeg', '.png')):
#         img_path = os.path.join(folder, img_name)
#         results = model(img_path)
#         results[0].save()  # This saves output in runs/detect/predict by default
#         print(f"Processed {img_name}")


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# import os

# folder = 'dental_data/data/rgb_images/cavity'

# print("Current Working Directory:", os.getcwd())
# print("Absolute Path to Folder:", os.path.abspath(folder))
# print("Folder Exists?:", os.path.exists(folder))


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# !ls /content/drive/MyDrive


In [ ]:
# !ls /content/drive/MyDrive


In [ ]:
# !ls dental_data
